# 🚀 Tool Calling Fine-Tuning Suite (LoRA / QLoRA / DoRA / Full FT)
Bu notebook, **Qwen2.5-0.5B** modeli üzerinde Tool / Function Calling yeteneğini eğitmek, değerlendirmek ve karşılaştırmak için hazırlanmıştır.

---

### 1. GPU Kontrolü

In [ ]:
!nvidia-smi

### 2. Proje Dizinini Ayarlama / Klonlama
> **Not:** GitHub reponuz varsa aşağıdaki linki güncelleyip klonlayabilirsiniz. Eğer dosyaları manuel yüklediyseniz `%cd tool-calling-ft` yapmanız yeterlidir.

In [ ]:
import os

# Eğer tool-calling-ft klasörü yoksa klonlayın:
if not os.path.exists("src/tool_calling_ft") and not os.path.exists("tool-calling-ft"):
    # Kendi GitHub repo adresinizi buraya yazabilirsiniz:
    # !git clone https://github.com/KULLANICI_ADI/tool-calling-ft.git
    pass

if os.path.exists("tool-calling-ft") and not os.path.exists("src/tool_calling_ft"):
    %cd tool-calling-ft

!pwd
!ls -la

### 3. Bağımlılıkların Kurulumu

In [ ]:
!pip install --upgrade pip
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install "transformers>=4.46" "peft>=0.13" "bitsandbytes>=0.44" "datasets>=3.0" "trl>=0.11" pandas pyyaml matplotlib pytest accelerate
!pip install --upgrade torchao
!pip install -e .

### 4. Veri Setini Hazırlama (Train / Val / Eval & Sentetik Negatifler)

In [ ]:
!python -m tool_calling_ft.data.prepare_dataset

### 5. Token Uzunluk Analizi

In [ ]:
!python scripts/measure_token_lengths.py

### 6. Fine-Tuning Eğitimi (QLoRA / LoRA / DoRA / Full FT)
İstediğiniz yöntemi aşağıdaki hücrelerden çalıştırabilirsiniz.

In [ ]:
# 🥇 1. QLoRA Eğitimi (4-bit NF4):
!python -m tool_calling_ft.training.train --config configs/qlora.yaml

In [ ]:
# 🥈 2. LoRA Eğitimi (16-bit):
# !python -m tool_calling_ft.training.train --config configs/lora.yaml

In [ ]:
# 🥉 3. DoRA Eğitimi (Weight-Decomposed LoRA):
# !python -m tool_calling_ft.training.train --config configs/dora.yaml

In [ ]:
# 🚀 4. Full Fine-Tuning:
# !python -m tool_calling_ft.training.train --config configs/full_ft.yaml

### 7. Model Değerlendirme (Evaluation Harness)
Eğitilen modellerin Tool Selection Accuracy, JSON Validity ve Argument Eşleşmesini test seti üzerinde ölçer.

In [ ]:
# 🔍 Ham Base Model (Baseline) Değerlendirmesi:
!python scripts/run_baseline.py

In [ ]:
# 🎯 QLoRA Modelini Değerlendirme:
!python -m tool_calling_ft.eval.harness --method qlora --adapter checkpoints/qlora --dataset data/processed/eval_subset.jsonl

In [ ]:
# 🎯 LoRA Modelini Değerlendirme:
# !python -m tool_calling_ft.eval.harness --method lora --adapter checkpoints/lora --dataset data/processed/eval_subset.jsonl

### 8. Canlı Test (Inference Demo)
Eğittiğiniz modele özel bir araç verip çıktısını canlı test edin.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

base_model_name = "Qwen/Qwen2.5-0.5B"
adapter_path = "checkpoints/qlora"

tokenizer = AutoTokenizer.from_pretrained(base_model_name)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
)
model = PeftModel.from_pretrained(model, adapter_path)
model.eval()

# Örnek test sorusu ve Tool şeması
system_prompt = """You are a function calling AI model. You are provided with function signatures within <tools> </tools> XML tags.
<tools>
[{"type": "function", "function": {"name": "get_current_weather", "description": "Get current weather for a city", "parameters": {"type": "object", "properties": {"location": {"type": "string"}, "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}}, "required": ["location"]}}}]
</tools>"""

user_query = "What is the weather in Tokyo in celsius?"

prompt = f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{user_query}<|im_end|>\n<|im_start|>assistant\n"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=128, eos_token_id=151645)

response = tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True)
print("="*50)
print("MODEL ÇIKTISI:")
print(response.strip())
print("="*50)

### 9. Metrik Karşılaştırma Tablosu

In [ ]:
import glob
import json
import pandas as pd

report_files = glob.glob("reports/*_metrics.json")
rows = []

for fpath in report_files:
    with open(fpath, "r", encoding="utf-8") as f:
        data = json.load(f)
    qm = data.get("quality_metrics", {})
    pm = data.get("performance_metrics", {})
    rows.append({
        "Method": data.get("method", "unknown"),
        "Tool Selection Acc": qm.get("tool_selection_accuracy", 0.0),
        "Argument Acc": qm.get("argument_accuracy", 0.0),
        "JSON Validity": qm.get("json_validity_rate", 0.0),
        "Negative Rejection": qm.get("negative_rejection_accuracy", 0.0),
        "Throughput (tokens/s)": pm.get("throughput_tokens_per_sec", 0.0),
        "Peak VRAM (MB)": pm.get("peak_vram_mb", 0.0),
    })

if rows:
    df = pd.DataFrame(rows)
    display(df.sort_values(by="Tool Selection Acc", ascending=False))
else:
    print("Henüz üretilmiş rapor bulunamadı.")

### 10. Sonuçları Yedekleme (Google Drive veya Zip İndirme)

In [ ]:
# Google Drive'a kopyalamak için:
from google.colab import drive
drive.mount('/content/drive')

!cp -r checkpoints /content/drive/MyDrive/tool_calling_checkpoints
!cp -r reports /content/drive/MyDrive/tool_calling_reports
print("✅ Checkpoint ve raporlar Drive'a kaydedildi!")

In [ ]:
# Alternatif olarak bilgisayarınıza zip indirmek için:
!zip -r tool_calling_results.zip checkpoints reports
from google.colab import files
files.download('tool_calling_results.zip')